# Threshold tools

This notebook demonstrates how to use `climakitae` to evaluate threshold exceedences for extreme weather events in downscaled climate model simulations. 

`climakitae` uses the Annual Maxima Series (AMS) method, as opposed to the alternative Peaks-Over-Threshold (POT) method, for extreme value analysis. In the AMS method, the maximum (or minimum) variable value is obtained for each year. A statistical distribution is fitted to the values and can then be used to obtain return probabilities, values, or periods. Users will have to option to choose their distribution along with other customizations.

**Terms used in this notebook**:
- __Return probabilities__: The probability of a threshold being exceeded within a given period of time. 
    - For example, the maximum temperature at a location has a 10% chance of exceeding 105°F in any year.
- __Return values__: A threshold expected to be exceeded at least once in a set period of time. 
    - A maximum temperature value that has a 10% annual return probability is the 10-year maximum temperature return value.
- __Return periods__: The frequency with which a threshold is expected to be exceeded at least once.
    - If the return probability of exceeding 150 mm of precipitation is 10%, the return period of that event is 10 years.

**Intended Application**: As a user, I want to learn how to:
- Generate return probabilities for climate simulations with climakitae
- Generate return values for climate simulations with climakitae
- Generate return periods for climate simulations with climakitae
- Visualize the spatial distribution of return value results across a region

**Runtime**: With the default settings, this notebook takes approximately X minutes to run from start to finish. Modifications to selections may increase the runtime.

**References**: The techniques in this notebook come from applications of extreme value theory to climate data. For further reading on this topic, see [Cooley 2009](https://link.springer.com/article/10.1007/s10584-009-9627-x).


In [ ]:
import climakitae as ck

## 1. Return value example for single location

In [ ]:
# Initialize the interface
cd = ck.ClimateData(verbosity=0)

# KFAT is Fresno, CA
one_in_x_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("prec")
    .processes(
        {
            "warming_level": {"warming_levels": [0.8, 2.0]},
            "convert_units": "inches/d",
            "clip": "KSAC",
            "metric_calc": {
                "one_in_x": {
                    "return_periods": [2, 5, 10],
                    "extremes_type": "max",
                }
            },
        }
    )
    .get()
)

In [ ]:
sims = [s.item() for s in one_in_x_data.sim if "EC-Earth3-Veg" not in s.item()]
one_in_x_data = one_in_x_data.sel(sim=sims)

In [ ]:
one_in_x_data.to_dataframe()[["return_values"]].groupby(
    ["warming_level", "one_in_x"]
).median()

In [ ]:
# Initialize the interface
cd = ck.ClimateData(verbosity=0)

# KFAT is Fresno, CA
one_in_x_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("prec")
    .processes(
        {
            "warming_level": {"warming_levels": [0.8, 2.0]},
            "convert_units": "inches/d",
            "clip": "KSAC",
            "metric_calc": {
                "one_in_x": {
                    "return_values": [1, 3, 5],
                    "extremes_type": "max",
                }
            },
        }
    )
    .get()
)

The EC-Earth3-Veg simulation does not have a complete warming level at 0.8, so we are dropping it from our analysis.

In [ ]:
sims = [s.item() for s in one_in_x_data.sim if "EC-Earth3-Veg" not in s.item()]
one_in_x_data = one_in_x_data.sel(sim=sims)

Display the return values ('one_in_x'), return periods and associated return probabilities, and p-values for this query.

In [ ]:
one_in_x_data.to_dataframe()[["return_periods", "return_probabilities"]].groupby(
    ["warming_level", "one_in_x"]
).median()

In [ ]:
one_in_x_data.to_dataframe()[["return_periods", "return_probabilities", "p_values"]]

## 2. Return period example with small region

TODO: Intro explaining the grouped_duration keyword (currenlty implemented to work with event_duration = 1 day).

In this case we are using a time framework to look at 3-day heat waves in a future period under SSP 3-7.0.

In [ ]:
# KFAT is Fresno, CA
one_in_x_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("t2max")
    .processes(
        {
            "time_slice": (2040, 2060),
            "convert_units": "degF",
            "clip": "San Bernardino County",
            "metric_calc": {
                "one_in_x": {
                    "return_values": [105],
                    "block_size": 1,
                    "extremes_type": "max",
                    "event_duration": (1, "day"),
                    "grouped_duration": (3, "day"),
                }
            },
        }
    )
    .get()
)

Map: Larger return periods indicate less frequent events. Areas with low return periods are more susceptible to 3-day heat waves 105F.

In [ ]:
one_in_x_data["return_periods"].median("sim").sel(one_in_x=105).plot(
    vmax=40, cmap="YlOrRd_r", x="lon", y="lat"
)

In [ ]:
one_in_x_data.to_dataframe()[["return_periods", "return_probabilities"]].groupby(
    ["one_in_x"]
).median()

## 3. Other ways to do 1-in-X analysis

Links to other notebooks.